# Exploring house prices in England and Wales
### Sumiyya and Syed · Sparta Global afternoon task

## Step 1 — Get and understand the data

We started by looking at the two questions in the task:

1. How might house prices change between now and next year for different property types and locations?
2. Given a house's size, location, number of bedrooms and other characteristics, what price would it likely sell for?

We wanted a dataset whose source and columns we could check, so we could explain why it was suitable before building a model. In this notebook, we introduce our chosen data, inspect a sample and identify what it can—and cannot—tell us.

## Why we chose this dataset

We chose **House Price per Square Metre in England and Wales**, created by UCL researchers and published on the **London Datastore**.

It links **HM Land Registry Price Paid Data**, which records completed sale prices, with **domestic Energy Performance Certificate (EPC) data**, which adds information such as floor area.

What made this useful for our task was having sale prices alongside size, location, property type and dates. We could check the publisher's documentation rather than rely only on a dataset title or an AI description.

- [Dataset description and 2024 download — London Datastore](https://data.london.gov.uk/dataset/house-price-per-square-metre-in-england-and-wales-epo9w)
- [Published field definitions](https://data.london.gov.uk/download/house-price-per-square-metre-in-england-and-wales/5313e02a-6f0c-4bca-8294-5be87a631160/Readme.pdf)
- [Full 2024 research dataset — UK Data Service](https://reshare.ukdataservice.ac.uk/857911/)

The 2024 version links source data through **31 October 2024**. Despite being hosted on London Datastore, its geographical coverage is England and Wales, not just London.

## How we made the data manageable

The full ZIP is about **1.24 GB**, and it contains separate CSV files for local authorities. We did not need to load everything to begin understanding it.

For this worked sample, we selected **Birmingham, Cardiff and Manchester**, keeping records whose source year is **2015–2024**. This gives us different locations to compare while keeping the analysis manageable. These are our selected examples, not a representative sample of the whole UK.

The bundled CSV preserves the source columns and adds a `location` label from each local-authority filename. Selecting a scope is preparation; we have not yet cleaned or trained a model on these records.

**Running this notebook:** extract the supplied ZIP and keep the `data` folder beside this notebook. Open it in Jupyter Notebook, JupyterLab or Google Colab. Run the code cells in order. In Colab, upload the data CSV into a folder named `data`.

In [2]:
# We load our selected records and check the size of the dataset.
from pathlib import Path
import pandas as pd
from IPython.display import display

data_file = Path('data/ucl_three_locations_2015_2024.csv')
if not data_file.exists():
    raise FileNotFoundError('Keep the supplied data folder beside this notebook before running it.')

housing = pd.read_csv(data_file, dtype={'postcode': str, 'lmk_key': str})
print(f'We have {housing.shape[0]:,} rows and {housing.shape[1]} columns in our selected sample.')

We have 209,903 rows and 17 columns in our selected sample.


## Our first look at the records

We displayed a few rows to connect the field names with actual examples. In this linked dataset, a row represents a **recorded sale linked to EPC information**. The same property can sell more than once, and the matching documentation distinguishes transactions linked to one or multiple EPCs. We will check repeated records and matching issues during cleaning.

Below, we show a few fields rather than all the identifying columns


In [3]:
# We show five records from each selected location.
preview_columns = ['location', 'dateoftransfer', 'propertytype', 'price', 'tfarea', 'numberrooms', 'duration']
preview = housing.groupby('location', sort=True).head(5)[preview_columns]
display(preview)

,location,dateoftransfer,propertytype,price,tfarea,numberrooms,duration
0,Birmingham,2017-06-22,F,55000,51.00,2.0,L
1,Birmingham,2023-12-21,S,250000,86.00,4.0,L
2,Birmingham,2019-01-22,T,162500,80.00,4.0,F
3,Birmingham,2021-02-26,F,150000,68.00,3.0,L
4,Birmingham,2023-01-13,F,60000,29.00,2.0,L
108155,Cardiff,2015-01-16,S,122500,68.62,3.0,F
108156,Cardiff,2023-03-24,T,195000,69.00,4.0,F
108157,Cardiff,2019-08-09,F,507500,107.00,NaN,L
108158,Cardiff,2021-09-29,T,220000,75.00,5.0,F
108159,Cardiff,2020-10-23,T,220100,44.10,3.0,F


## What the main columns mean

We checked the field definitions before deciding how the columns might help answer the task.

| Column | What it means | Why it matters to us |
|---|---|---|
| `price` | Recorded completed sale price in pounds | The outcome we could predict for an individual sale |
| `tfarea` | Total floor area in square metres | Lets us investigate the relationship between size and price |
| `propertytype` | D: detached; S: semi-detached; T: terraced; F: flat/maisonette | Lets us compare property types |
| `postcode` | Property postcode | Gives us more detailed location information |
| `location` | Local-authority label we added from the source filename | Identifies our three selected locations |
| `dateoftransfer` | Date the sale was completed | Lets us study dates and later plan a chronological split |
| `duration` | F: freehold; L: leasehold | Another property characteristic that may relate to price |
| `numberrooms` | Number of habitable rooms | Potential characteristic, but **not a bedroom count** |
| `priceper` | Recorded price divided by floor area | Useful descriptively, but it contains the price we want to predict |
| `classt` | Class describing the transaction-to-EPC matching relationship | Helps us investigate matching quality during cleaning |

We will not use `priceper` to predict `price`, because it already contains the answer. Transaction IDs and certificate keys help with checks, but are not meaningful price predictors.

In [7]:
# We inspect how the columns have been read into Python.
# Reading a date as text is common; we will convert it during step 2.
column_overview = housing.dtypes.astype(str).rename('Data type').to_frame()
column_overview.index.name = 'Column'
display(column_overview)

,Data type
Column,
priceper,float64
year,int64
dateoftransfer,str
propertytype,str
duration,str
price,int64
postcode,str
lad23cd,str
transactionid,str


## Checking the locations, dates and property types

We checked the coverage rather than assuming every location had the same number of sales or the same date range. This also helps us avoid treating an incomplete year as a complete one.

In [8]:
# We create a temporary parsed date for inspection; we do not alter the source column.
inspection = housing.assign(inspection_date=pd.to_datetime(housing['dateoftransfer'], errors='coerce'))
coverage = inspection.groupby('location').agg(
    Recorded_rows=('price', 'size'),
    First_sale=('inspection_date', 'min'),
    Last_sale=('inspection_date', 'max')
)
display(coverage)

type_names = {'D': 'Detached', 'S': 'Semi-detached', 'T': 'Terraced', 'F': 'Flat/maisonette'}
property_counts = pd.crosstab(housing['location'], housing['propertytype']).rename(columns=type_names)
display(property_counts)

,Recorded_rows,First_sale,Last_sale
location,,,
Birmingham,108155,2015-01-02,2024-10-29
Cardiff,46547,2015-01-01,2024-10-25
Manchester,55201,2015-01-02,2024-10-30


propertytype,Detached,Flat/maisonette,Semi-detached,Terraced
location,,,,
Birmingham,11587,23082,35442,38044
Cardiff,6796,10536,12060,17155
Manchester,2310,20518,15032,17341


## How the data fit our two questions

**Predicting an individual sale price:** we have recorded prices, floor area, property type, location and other characteristics to investigate. We do **not** have a verified bedroom count. We will not describe habitable rooms as bedrooms or claim that this fully answers the bedroom-based part of the task.

**Exploring next-year changes:** the sale dates give us historical information, but the extract stops in 2024. We cannot call its last prices “today's prices.” We plan to supplement it with more recent **UK House Price Index** data for our chosen locations and property types. We have not made a forecast in step 1.

[Official UK House Price Index information and methodology](https://www.gov.uk/government/publications/about-the-uk-house-price-index/about-the-uk-house-price-index)

## What we need to be careful about

- **Coverage:** our worked sample contains three selected local authorities in England and Wales. We cannot generalise it to every UK property.
- **Bedrooms:** habitable rooms and bedrooms are different measurements.
- **Dates:** the source ends in October 2024, so 2024 is not a complete year.
- **Linking:** records were matched across two sources. We still need to examine missing values, repeated transactions and unusual measurements.
- **Information available at sale:** a linked certificate could have been issued after the sale or after changes to the property. These limited fields alone do not establish that the EPC information was available before each sale. We must investigate timing before claiming a leakage-free historical prediction.
- **Recorded sales:** the prices describe completed transactions, not every home or current asking price.

Finding these limitations is part of understanding the data; it does not mean we should hide them or invent missing information.

## Our step-1 takeaway

We have identified the source, inspected real records, checked our selected coverage and clarified the important fields. We now have a clearer idea of which parts of the task the data support and where the gaps are.

Our next step is to **explore and clean the data**: check missing values, duplicates, data types and unusual prices or floor areas, then record any changes we make.

Before we move on, we will each explain one record and one limitation to each other so we both understand the starting point.